In [1]:
from scipy.sparse import lil_matrix

In [2]:
%load_ext cython

In [6]:
%%cython
cimport cython
cimport numpy as cnp
import numpy as np
from cmath import sqrt

@cython.boundscheck(False)
cdef assemblenxn(int n,C0,Cl,Cr,C2,dzm,dzp,lil,periodic=False):
    assert not periodic, "Periodic not written yet"
    cdef:
        int i,nz,z
        list rinds, rdata
    nz=C0.shape[-1]
    for r,(rinds,rdata) in enumerate(zip(lil.rows,lil.data)):
        i=r%n
        z=int((r-i)/2)
        
        #put in left
        if z>0:
            rinds+=range(n*z-n,n*z)
            rdata+=(\
                -C2[i,:n,z-1]/dzp[z-1]/sqrt(dzm[z]*dzm[z-1])\
                -1j*(Cl[i,:n,z]+Cr[i,:n,z-1])/sqrt(dzm[z]*dzm[z-1])).tolist()
        
        #put in diag
        rinds+=range(n*z,n*z+n)
        rdata+=(\
             C0[i,:n,z]+\
            (C2[i,:n,z-1]/dzp[z-1]/dzm[z] if z>0    else 0) +\
            (C2[i,:n,z  ]/dzp[z  ]/dzm[z] if z<nz-1 else 0)).tolist()
        
        #put in right
        if z<nz-1:
            rinds+=range(n*z+n,n*z+2*n)
            rdata+=(\
                -C2[i,:n,z]/dzp[z]/sqrt(dzm[z]*dzm[z+1])\
                -1j*(Cl[i,:n,z]+Cr[i,:n,z+1])/sqrt(dzm[z]*dzm[z+1])).tolist()
    

In [7]:
dzm=u=np.ones((5))
dzp=um=np.ones((4))
C0=np.array([[1*u,2*u],[3*u,4*u]])
C2=np.array([[10*um,20*um],[30*um,40*um]])
Cl=np.array([[100*u,200*u],[300*u,400*u]])
Cr=np.array([[1000*u,2000*u],[3000*u,4000*u]])
lil=lil_matrix((2*len(u),2*len(u)),dtype='complex')
assemblenxn(2,C0,Cl,Cr,C2,dzm,dzp,lil,periodic=False)
lil.todense()

NameError: name 'assemblenxn' is not defined

In [8]:
lil=lil_matrix((2*len(u),2*len(u)),dtype='complex')

In [10]:
lil.rows[0]=[1,2,]

In [11]:
lil.rows

array([[1, 2], [], [], [], [], [], [], [], [], []], dtype=object)